# Starting with rag
## Requirements :
    -langchain-community
    -PyPDF
    -PymuPDF


### loading documents

In [ ]:
# Text Loader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../doc_files/notes.txt")
content = loader.load()
print(content)


In [ ]:
# Directory Loader
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import JSONLoader

dirLoader = DirectoryLoader(
    "../doc_files/",
    glob="**/*.json",
    loader_cls=JSONLoader,
    loader_kwargs={"jq_schema": ".", "text_content": False},
    show_progress=False
)
content = dirLoader.load()
content

In [ ]:
# Loading Pdf File
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../doc_files/",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    loader_kwargs={"extract_images":False, "extract_tables": "markdown"},
    use_multithreading=True
)
contents = dir_loader.load()
contents

In [ ]:
# Create dummy files and write the some content 

import os

storage = {
    "notes.txt": "This is a simple text file content.",
    "config.json": '{"setting": "enabled", "version": 1.0}',
    "script.py": "print('Hello from the script!')",
}

for file, content in storage.items():
    with open(f"../doc_files/{file}", "w", encoding="utf-8") as f:
        f.write(content)


print("content Written Successfully...")

# RAG Pipeline (From Indexing to Vector Db pipleine)
### Requirements: 
    -langchain-community(PyPDFLoader and PyMuPDF)
    -langchain.textsplitter (RecurisveCharacterTextSplitter)
    pathlib

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [ ]:
"""
create a function that loads all the pdf file in the dir and 
returns the whole documents by adding corresponding
metadata fields like file_name and filetype
"""
def getPdfDocs(pdfDir):
    allDocs = []
    pobj = Path(pdfDir)
    if not pobj.is_dir():
        print(f"Dir Not Found")
        return None
    print(pobj.rglob("**/*.pdf"))
    pdf_files = pobj.rglob("**/*.pdf")
    # procces the pdf files 
    for pdf_file in pdf_files:
        print(f"Processing :{pdf_file}")
        loader = PyPDFLoader(str(pdf_file))
        docs = loader.load()
        print(f"Loaded {len(docs)} pages")
        for doc in docs:
            doc.metadata["file_name"] = pdf_file.stem
            doc.metadata["file_type"] = pdf_file.suffix
        allDocs.extend(docs)
    return allDocs
all_docs = getPdfDocs("../doc_files/")

In [ ]:
# a text splitter function
def split_doc(docs, chunkSize=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunkSize,
        chunk_overlap = chunk_overlap,
        separators=["\n\n", "\n", " ", ""],
        length_function =len
    )
    splitted_doc = text_splitter.split_documents(docs)
    print(f"splitted {len(docs)} Docs into {len(splitted_doc)}")
    # print(f"Content: {splitted_doc[0].page_content[:200]}")
    return splitted_doc
splitted_docs = split_doc(all_docs)
splitted_docs

# Embedding and vectordb

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
import uuid
from typing import List, Dict, Any, Tuple
import numpy as np
import os

In [ ]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = None
        self.model_name = model_name
        self._load_model()
    
    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Dimension : {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading Model: {self.model_name} : {e}")
            raise ValueError("Model Cannot Be Loaded...")
        
    def generate_embedding(self, texts: List[str]):
        try:
            encodings = self.model.encode(texts, show_progress_bar=True)
            return encodings
        except Exception as e :
            print(e)
            
    def get_embedding_dimension(self):
        if not self.model:
            raise ValueError("Model Doesn't Exist")
        return self.model.get_embedding_dimension()

embedding_manager = EmbeddingManager()
embedding_manager

In [ ]:
# Vector Store
class VectorStore:
    def __init__(self, collection_name: str = "PDF_Collection", persist_dir: str = "./vector_store"):
        self.collection_name = collection_name
        self.persist_dir = persist_dir
        self.collection = None
        self.client = None
        self._load_vectorDB()
    
    # load the colleciton and client
    def _load_vectorDB(self):
        # initialize collection and client
        os.makedirs(self.persist_dir, exist_ok=True)
        try:
            self.client = chromadb.PersistentClient(path=self.persist_dir)
            self.client.delete_collection(name=self.collection_name)
            self.collection = self.client.get_or_create_collection(
                self.collection_name, 
                configuration={
                  "hnsw": {
                      "space": "cosine"
                  }
                },
                metadata={
                    "description": "PDF files for RAG"
                    }
                )
            print(f"Vector Store initialized successfully {self.collection_name}")
            print(f"Existing Documents in collection {self.collection.count()}")
        except Exception as e:
            print(e)    
    # Add docs to vector store 
    def add_docs(self, docs: List[Any], embeddings: np.ndarray):
        # prepare data 
        doc_ids = []
        doc_contents = []
        doc_embeddings = []
        metadatas = []
        for i, (doc, embedding) in enumerate(zip(docs, embeddings)):
            # unique id for each doc
            uid = f"{uuid.uuid4().hex[:8]}_{i}"
            doc_ids.append(uid)
            
            # prepare page content
            doc_contents.append(doc.page_content)
            
            # embeddings 
            doc_embeddings.append(embedding.tolist())
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["document_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)
            print(f"metadata: {metadata}")
            
        # add fields to vector Store 
        try:
            self.collection.add(
                ids=doc_ids,
                documents=doc_contents,
                embeddings=doc_embeddings,
                metadatas=metadatas
            )
            print(f"Successfully added {len(docs)} into vector store...")
        except Exception as e :
            print(e)


vectorstore_manager = VectorStore()

In [ ]:
# doc = Document(
#     page_content="Hello, LangChain!",
#     metadata={"source": "manual_input"}
# )
# emb = embedding_manager.generate_embedding([doc.page_content])
# emb.tolist()
# vectorstore_manager.add_docs(docs=[doc], embeddings=emb)


In [ ]:
# Extract the texts from page content 
texts = [doc.page_content for doc in splitted_docs]

# Generate Embedding 
embeddings = embedding_manager.generate_embedding(texts)

# store embedding and chunks into vector store 
vectorstore_manager.add_docs(splitted_docs, embeddings)

In [ ]:
# Reterival Pipeline
class ReterivalManager:
    def __init__(self, embedding_manager: EmbeddingManager, vector_store: VectorStore):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store
    
    def reteriveContext(self, query: str, top_k: int, threshold: float = 0.0) -> list[Dict[str, Any]]:
        # Generate the qurey embedding 
        query_embedding = self.embedding_manager.generate_embedding([query])
        
        # serach in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=query_embedding.tolist(),
                n_results=top_k
            )
            reterived_docs = []
            if results["documents"] and results["documents"][0]:
                docs = results["documents"][0]
                ids = results["ids"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                
                for i, (id_, doc, metadata, distance) in enumerate(zip(ids, docs, metadatas, distances)):
                    # convert distanbce to simlarity score (Chromadb uses cosine distance)
                    similarity_score = 1 - distance
                    if similarity_score >= threshold:
                        reterived_docs.append({
                            "id": id_,
                            "content": doc,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                print(f"Reterived {len(reterived_docs)}")
            else:
                print("no document Reterived ")
            return reterived_docs
        except Exception as e:
            print(e)

reterival = ReterivalManager(embedding_manager, vectorstore_manager)

In [ ]:
print(vectorstore_manager.collection.count())
print(vectorstore_manager.collection.configuration)
print(vectorstore_manager.collection.get())
print(vectorstore_manager.collection.count())

In [ ]:
results = vectorstore_manager.collection.get(
    include=["documents", "metadatas"]
)

for i, (doc, metadata) in enumerate(
    zip(results["documents"], results["metadatas"])
):
    print(f"\n--- CHUNK {i} ---")
    print(doc)
    print("METADATA:", metadata)

In [ ]:
results = reterival.reteriveContext(
    query="Conclusion of the Srinivas Premier League Season ",
    top_k=10
)

for result in results:
    print(
        f"\nRank: {result['rank']}"
        f"\nSimilarity: {result['similarity_score']}"
        f"\nContent: {result['content'][:200]}"
    )

In [ ]:
results = reterival.reteriveContext(
    query="what are the skills of vivekanand",
    top_k=10
)

for result in results:
    print(
        f"\nRank: {result['rank']}"
        f"\nSimilarity: {result['similarity_score']}"
        f"\nContent: {result['content'][:200]}"
    )

In [ ]:
# llm
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b",temperature=0.0, max_tokens=1024, timeout=60, max_retries=2)
llm

In [ ]:
# Create Enhanced RAG pipeline 

# Advance RAG Pipeline 
    - Citations
    - Streaming
    - History
    - Summarization

In [ ]:
import re
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage, AIMessage

 
class AdvanceRAG:
    def __init__(self, reteriver: ReterivalManager, llm):
        self.reteriver = reteriver
        self.llm = llm
        self.history = []
        self.summarizer = SummarizationMiddleware(
            model=llm,
            trigger=("tokens", 10000),
            keep=("messages", 4)
        )
        
    
    # Query functrion to produce answer from retrieved docs
    def query_func(self, query: str, top_k: int, min_score: float = 0.0, stream: bool = False, summarize: bool = False):
        # retrive the document
        retrived_doc = self.reteriver.reteriveContext(query=query, top_k=top_k, threshold=min_score)
        answer = ""
        if not retrived_doc:
            print("No Context Found ")
            context = ""
            sources = []
            answer = "No relevant chunk found"
        else:
            # prepare context along with citation label
            context = "\n\n".join(
                f"[{i}]\n{doc['content']}" for i, doc in enumerate(retrived_doc, start=1)
            )
            print("\n========== CONTEXT SENT TO LLM ==========\n")
            print(context)
            print("\n==========================================\n")
            # prepare sources
            sources = [{
                "citation_id": i,
                "chunk_id": doc["id"],
                "source": doc["metadata"].get("source", doc["metadata"].get("source_file", "unknown")),
                "page": doc["metadata"].get("page", "unknown"),
                "score": doc["similarity_score"],
                "preview": doc["content"][:120] + "..."
            } for i, doc in enumerate(retrived_doc,  start=1) ]
            
            history_context = "\n".join(
                f"{message.type}: {message.content}"
                for message in self.history
            )
            
            prompt = f"""
            You are a RAG assistant.

            Use the conversation history only to understand the user's current question.
            Use the retrieved context as the source of truth for factual answers.

            Rules:

            - Do not use outside knowledge.
            - If the answer is not in the context, say:
            "I don't have enough information in the provided documents."
            - Cite factual statements using ONLY [1], [2], [3].
            - NEVER use any other citation format such as 【1】, (1), or [1†L1-L4].
            - Use only citation numbers present in the context.
            - Put citations immediately after the supported statement.

            Conversation History:
            {history_context}

            Context:
            {context}

            Question:
            {query}
            """
                        
                        
            # stream the resp (if Stream)   
            if stream:
                # stream with citations(answer with citatiosn)
                for chunk in self.llm.stream(prompt):
                    if (len(chunk.content) != 0):
                        answer += chunk.content
                        print(chunk.content, flush=True, end="")
            else:
                response = self.llm.invoke(prompt)
                answer = response.content
            print("\n========== LLM ANSWER ==========\n")
            print(repr(answer))
            print("\n================================\n")
            
            self.history.append(
                HumanMessage(content=query)
            )

            self.history.append(
                AIMessage(content=answer)
            )
            
            # Citations
            citation_map = {
                source["citation_id"] : source
                for source in sources
            }
            used_citations = sorted({
                int(num)
                for num in re.findall(r"【(\d+)】", answer)
            })
            used_sources = [
                citation_map[c_num]
                for c_num in used_citations
                if c_num in citation_map
            ]
            print("used_sources: ", used_sources)
            
            return {
                "question": query,
                "answer": answer,
                "sources": used_sources
            }
                        
            # Summarize 
            
            # Store History(question, answer, sources, summary) 
            
            # return {'question', 'answer', 'sources', 'summary', 'history'}
        

rag = AdvanceRAG(llm=llm, reteriver=reterival)
rag

In [ ]:
import re
from typing import Any, Dict, List, Optional, Tuple
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware


class AdvanceRAG:

    def __init__(self, reteriver: Any, agent: Any, llm: Optional[Any] = None):
        self.reteriver = reteriver
        self.agent = agent
        self.llm = llm

        # Stores raw conversation messages: [HumanMessage, AIMessage, ...]
        self.history: List[Any] = []

    # ---------------------------------------------------------
    # 1. Retrieve relevant documents
    # ---------------------------------------------------------
    def _retrieve(self, query: str, top_k: int, min_score: float) -> List[Dict]:
        return self.reteriver.reteriveContext(
            query=query, top_k=top_k, threshold=min_score
        )

    # ---------------------------------------------------------
    # 2. Build RAG context with explicit bracket styling
    # ---------------------------------------------------------
    def _build_context(
        self, retrieved_docs: List[Dict]
    ) -> Tuple[str, List[Dict]]:

        # insert Citations to context
        context = "\n\n".join(
            f"[{i}]\n{doc['content']}"
            for i, doc in enumerate(retrieved_docs, start=1)
        )

        sources = [
            {
                "citation_id": i,
                "chunk_id": doc.get("id", f"chunk_{i}"),
                "source": doc.get("metadata", {}).get(
                    "source",
                    doc.get("metadata", {}).get("source_file", "unknown"),
                ),
                "page": doc.get("metadata", {}).get("page", "unknown"),
                "score": doc.get("similarity_score"),
                "preview": doc["content"][:120] + "...",
            }
            for i, doc in enumerate(retrieved_docs, start=1)
        ]

        return context, sources

    # ---------------------------------------------------------
    # 3. Build System & User Messages
    # ---------------------------------------------------------
    def _build_messages(
        self, query: str, context: str
    ) -> Tuple[SystemMessage, HumanMessage]:
        system_content = f"""You are a RAG assistant.
Use the conversation history only to understand the user's current question.
Use the retrieved context as the ONLY source of factual information.

Rules:
- Do not use outside knowledge.
- If the answer is not present in the retrieved context, say:
  "I don't have enough information in the provided documents."
- Cite factual statements using ONLY square brackets matching the context (e.g., [1], [2]).
- NEVER use any other citation format such as 【1】, (1), or [1†L1-L4].
- Put citations immediately after the supported statement.
- Do not invent citations.

Retrieved Context:
{context}"""

        return SystemMessage(content=system_content), HumanMessage(
            content=query
        )

    # ---------------------------------------------------------
    # 4. Extract citations from answer (Supports [1] and 【1】)
    # ---------------------------------------------------------
    def _extract_citations(self, answer: str) -> List[int]:
        matches = re.findall(r"\[(\d+)\]|【(\d+)】", answer)
        return sorted({int(a or b) for a, b in matches})

    # ---------------------------------------------------------
    # 5. Resolve citation numbers to actual sources
    # ---------------------------------------------------------
    def _resolve_sources(
        self, answer: str, sources: List[Dict]
    ) -> List[Dict]:
        citation_map = {source["citation_id"]: source for source in sources}
        used_citations = self._extract_citations(answer)

        return [
            citation_map[citation_id]
            for citation_id in used_citations
            if citation_id in citation_map
        ]

    # ---------------------------------------------------------
    # Main Query Method
    # ---------------------------------------------------------
    def query_func(
        self,
        query: str,
        top_k: int = 4,
        min_score: float = 0.0,
        stream: bool = True,
    ) -> Dict[str, Any]:

        # 1. Retrieval
        retrieved_docs = self._retrieve(
            query=query, top_k=top_k, min_score=min_score
        )

        if not retrieved_docs:
            print("\n[INFO] No Context Found")
            answer = "I don't have enough information in the provided documents."

            self.history.append(HumanMessage(content=query))
            self.history.append(AIMessage(content=answer))

            return {
                "question": query,
                "answer": answer,
                "sources": [],
            }

        # 2. Build Context & Sources
        context, sources = self._build_context(retrieved_docs)

        print("\n========== CONTEXT SENT TO LLM ==========\n")
        print(context)
        print("\n==========================================\n")

        # 3. Construct System Prompt & User Message
        system_msg, user_msg = self._build_messages(query, context)

        # Build execution payload (System Prompt + Full Conversation History + Current Query)
        input_messages = [system_msg] + self.history + [user_msg]

        # 4. Model Generation
        answer = ""

        if stream:
            if self.agent is None:
                raise ValueError("llm must be provided when stream=True.")

            print("\n========== LLM ANSWER (STREAMED) ==========\n")
            for chunk in self.agent.stream({"messages": input_messages}):
                print("chunk: ", chunk)
                content = getattr(chunk, "content", str(chunk))
                if content:
                    answer += content
                    print(content, end="", flush=True)
            print("\n==========================================\n")

        else:
            response = self.agent.invoke({"messages": input_messages})

            # Handle response formats from standard LangChain Agents
            if isinstance(response, dict) and "messages" in response:
                answer = response["messages"][-1].content
            elif hasattr(response, "content"):
                answer = response.content
            else:
                answer = str(response)

            print("\n========== LLM ANSWER ==========\n")
            print(answer)
            print("\n=================================\n")

        # 5. Resolve Used Citations
        used_sources = self._resolve_sources(answer=answer, sources=sources)
        print("used_sources:", used_sources)

        # 6. Update History cleanly (User Query & AI Answer only)
        self.history.append(HumanMessage(content=query))
        self.history.append(AIMessage(content=answer))

        return {
            "question": query,
            "answer": answer,
            "sources": used_sources,
            "history": self.history,
        }
   
summarizer = SummarizationMiddleware(
    model=llm,
    trigger=("tokens", 10000),
    keep=("messages", 4)
)
agent = create_agent(
    model=llm,
    tools=[],
    middleware=[summarizer]
)     
rag = AdvanceRAG(reterival, agent, llm)    


In [ ]:
result = rag.query_func(
    query="Who orgainzed spl",
    top_k=5
)

print("\n\nANSWER:")
print(result["answer"])

print("\n\nSOURCES:")
print(result["sources"])
for source in result["sources"]:
    print(source)

In [ ]:
result = rag.query_func(
    query="What are his Strength",
    top_k=5
)

print("\n\nANSWER:")
print(result["answer"])

print("\n\nSOURCES:")
print(result["sources"])
for source in result["sources"]:
    print(source)

In [ ]:
import re

answer = resp

used_citations = [
    int(number)
    for number in set(re.findall(r"\[(\d+)\]", answer.get("answer").content))
]

print("Used citations:", used_citations)

# get the valid Citations
valid_citations = {
    source["citation_id"]
    for source in answer.get("sources")
}

print("Valid citations:", valid_citations)

In [ ]:
result

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from typing import Any, List, Tuple

class AdvanceRagImplementation:
    # init
    def __init__(self, agent: Any, reteriver: ReterivalManager):
        self.agent = agent
        self.reteriver = reteriver
        self.history = []
        
    # reterival
    def _reterive_doc(self, query: str, top_k: int, threshold: float)->list[Dict[str, Any]]:
        return self.reteriver.reteriveContext(query=query, top_k=top_k, threshold=threshold)
    
    # context with citations and source 
    def _build_context(self, reterived_doc: Any) -> Tuple[str, List[Dict]]:
        context = "".join(
            f"[{i}]\n{doc.content}"
            for i, doc in enumerate(reterived_doc, start=1))
        
        sources = [
            {
                "citation_id": i,
                "chunk_id": doc.get("id", f"chunk_{i}"),
                "source": doc.get("metadata", {}).get(
                    "source",
                    doc.get("metadata", {}).get("source_file", "unknown"),
                ),
                "page": doc.get("metadata", {}).get("page", "unknown"),
                "score": doc.get("similarity_score"),
                "preview": doc["content"][:120] + "...",
            }
            for i, doc in enumerate(reterived_doc, start=1)
        ]
        return context, sources
    
    # Build Input (sys msg, history , question/query)
    def _build_messages(self, context: str, query: str) ->Tuple[Any, Any]:
        sys_prompt = f"""
        prompt...
        {context}
        """
        return SystemMessage(content=sys_prompt), HumanMessage(content=query)
    
    # Extract the cittaions from the answer 
    def _extract_citations(self, answer: str) -> List[int]:
        matches = re.findall(r"\[(\d+)\]|【(\d+)】", answer)
        return sorted({int(a or b) for a, b in matches})
    
    def _resolve_sources(self, answer: str, sources: Dict):
        citation_map = [
            {
                source["citation_id"]: source 
                for source in sources
            }
        ]
        used_citations = self._extract_citations(answer)
        return 
    def query_fun(self, query: str, top_k: int, threshold: float, stream: bool = False, summarize: bool = False)-> tuple[Dict, Dict, Dict]:
        # 1. reteriver
        reterived_doc = self._reterive_doc(query, top_k, threshold)
        
        # 2. build context and sources from reterived_doc
        context, sources = self._build_context(reterived_doc)
        
        # 3. set a fallback technique in case for no reterived doc 
        if not context:
            print("No Context found")
            answer = ""
            sources = []
            
        
        # 4. Buid the Message(System Message and Human Message)
        sys_message, human_message = self._build_messages(context, query)
        
        # 5. Build the execution payload (combine history + SystemMessage + query)
        input_message = [sys_message] + self.history + [human_message]
        
        # 6. Model Generation 
        if stream:
            for chunk in self.agent.stream({"messages", input_message}):
                if chunk:
                    answer += chunk.content
                    print(chunk.content, end = "", flush=True)
        else:
            resp = self.agent.invoke({"messages": input_message})
            if isinstance(resp, (Dict, dict)) and "messages" in resp:
                answer = resp["messages"][-1].content
            if hasattr(resp, "content"):
                answer = resp.content
            else:
                answer = str(resp)
        
        # 7. resolve used sources from the citations 
        used_sources = self._resolve_sources(answer, sources)
        
        # 8. update the history 
        self.history.append(sys_message)
        self.history.appenf(human_message)
        
        # 7. return eturn {'question', 'answer', 'sources', 'history'}
        return {
            "question": query,
            "answer": answer,
            "source": used_sources,
            "history": self.history
        }


SyntaxError: incomplete input (2858172120.py, line 29)